# Notebook 3 — Análises e Visualizações
**Tech Challenge Fase 3 · Big Data to Analytics · POSTECH**

Geração dos gráficos analíticos a partir das tabelas Gold do S3.

**Análises cobertas:**
1. Diversidade de gênero (evolução 2021–2023)
2. Top cargos no mercado de dados
3. Distribuição por senioridade
4. Faixa salarial por senioridade (2023)
5. Modelo de trabalho
6. Diversidade racial
7. Linguagens de programação mais usadas
8. Adoção de Cloud (AWS, Azure, GCP)
9. Ferramentas de BI
10. Adoção de IA Generativa (2023)

## 1. Setup — Leitura dos dados Gold via Spark

In [ ]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import subprocess

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

BUCKET = 's3://tc-fase3-state-of-data-wl'
GOLD   = f'{BUCKET}/gold'

# Paleta de cores
AZUL_ESCURO = '#0d1b2a'
AZUL_CLARO  = '#00c9ff'
AMARELO     = '#f0c040'
VERDE       = '#4ade80'
ROSA        = '#f472b6'
LARANJA     = '#fb923c'
ROXO        = '#a78bfa'
CINZA       = '#94a3b8'
BRANCO      = '#ffffff'

def estilo(ax, titulo, xlabel='', ylabel=''):
    ax.set_facecolor(AZUL_ESCURO)
    ax.set_title(titulo, color=BRANCO, fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel(xlabel, color=CINZA, fontsize=9)
    ax.set_ylabel(ylabel, color=CINZA, fontsize=9)
    ax.tick_params(colors=CINZA)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    for sp in ['bottom','left']: ax.spines[sp].set_color('#1e3a5f')
    for l in ax.get_xticklabels()+ax.get_yticklabels(): l.set_color(CINZA)

print('Setup concluído!')

## 2. Diversidade de Gênero (2021–2023)

In [ ]:
df_genero = spark.read.parquet(f'{GOLD}/diversidade_genero/').toPandas()
print(df_genero.sort_values(['ano_pesquisa','total'], ascending=[True,False]))

anos = ['2021','2022','2023']
masc  = [81.1, 75.0, 75.1]
fem   = [18.6, 24.8, 24.4]
outro = [0.3,  0.3,  0.5]

x = np.arange(len(anos)); w = 0.28
fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor(AZUL_ESCURO)
b1 = ax.bar(x - w, masc,  w, label='Masculino', color=AZUL_CLARO, alpha=0.9)
b2 = ax.bar(x,     fem,   w, label='Feminino',  color=ROSA,        alpha=0.9)
b3 = ax.bar(x + w, outro, w, label='Outro/NI',  color=AMARELO,     alpha=0.9)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', color=BRANCO, fontsize=9, fontweight='bold')
estilo(ax, 'Diversidade de Gênero (2021–2023)', ylabel='% dos Respondentes')
ax.set_xticks(x); ax.set_xticklabels(anos, color=BRANCO, fontsize=11)
ax.set_ylim(0, 95); ax.legend(facecolor='#1e3a5f', labelcolor=BRANCO)
ax.yaxis.grid(True, color='#1e3a5f', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/tmp/grafico_genero.png', dpi=150, bbox_inches='tight', facecolor=AZUL_ESCURO)
plt.show()
subprocess.run(['aws','s3','cp','/tmp/grafico_genero.png',f'{BUCKET}/graficos/'])
print('Gráfico gênero salvo!')

## 3. Senioridade por ano

In [ ]:
niveis = ['Júnior','Pleno','Sênior']
s2021 = [621, 657, 576]; s2022 = [1023,1060,898]; s2023 = [1046,1392,1418]
x = np.arange(len(niveis)); w = 0.25
fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor(AZUL_ESCURO)
b1 = ax.bar(x-w, s2021, w, label='2021', color=AZUL_CLARO, alpha=0.85)
b2 = ax.bar(x,   s2022, w, label='2022', color=VERDE,       alpha=0.85)
b3 = ax.bar(x+w, s2023, w, label='2023', color=AMARELO,     alpha=0.85)
for bars in [b1,b2,b3]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                str(int(bar.get_height())), ha='center', va='bottom', color=BRANCO, fontsize=8.5, fontweight='bold')
estilo(ax, 'Distribuição por Senioridade (2021–2023)', ylabel='Nº de Respondentes')
ax.set_xticks(x); ax.set_xticklabels(niveis, color=BRANCO, fontsize=12)
ax.set_ylim(0,1700); ax.legend(facecolor='#1e3a5f', labelcolor=BRANCO)
ax.yaxis.grid(True, color='#1e3a5f', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/tmp/grafico_senioridade.png', dpi=150, bbox_inches='tight', facecolor=AZUL_ESCURO)
plt.show()
subprocess.run(['aws','s3','cp','/tmp/grafico_senioridade.png',f'{BUCKET}/graficos/'])
print('Gráfico senioridade salvo!')

## 4. Linguagens de Programação (2021–2023)

In [ ]:
# Dados obtidos via Athena — query de tecnologias
anos    = ['2021','2022','2023']
totais  = [2641, 4270, 5293]
linguagens = {
    'Python':     [1344, 2089, 2826],
    'SQL':        [1484, 2367, 3155],
    'R':          [305,  385,  408],
    'Java':       [222,  263,  355],
    'JavaScript': [173,  0,    282],
    'Scala':      [89,   119,  131],
}
cores = [AZUL_CLARO, AMARELO, VERDE, ROSA, LARANJA, ROXO]

fig, ax = plt.subplots(figsize=(10, 5.5))
fig.patch.set_facecolor(AZUL_ESCURO)
x = np.arange(len(anos)); n = len(linguagens); w = 0.13
offset = -(n-1)/2 * w
for i, (lang, vals) in enumerate(linguagens.items()):
    pcts = [v/t*100 for v,t in zip(vals, totais)]
    bars = ax.bar(x+offset+i*w, pcts, w, label=lang, color=cores[i], alpha=0.9)
    for bar, pct in zip(bars, pcts):
        if pct > 2:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    f'{pct:.0f}%', ha='center', va='bottom', color=BRANCO, fontsize=6.5, fontweight='bold')
estilo(ax, 'Linguagens Mais Usadas pelos Profissionais de Dados (2021–2023)', ylabel='% dos Respondentes')
ax.set_xticks(x); ax.set_xticklabels(anos, color=BRANCO, fontsize=11)
ax.set_ylim(0,75); ax.yaxis.grid(True, color='#1e3a5f', linestyle='--', alpha=0.5)
ax.legend(facecolor='#1e3a5f', labelcolor=BRANCO, fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/grafico_linguagens.png', dpi=150, bbox_inches='tight', facecolor=AZUL_ESCURO)
plt.show()
subprocess.run(['aws','s3','cp','/tmp/grafico_linguagens.png',f'{BUCKET}/graficos/'])
print('Gráfico linguagens salvo!')

## 5. Adoção de IA Generativa (2023)

In [ ]:
# Dados obtidos via Athena — query de adoção de IA
# 2023: nao_usa=743, usa_gratis=2270, usa_pago=274, empresa_paga=232, total=3772
ia_labels = ['Não usa IA', 'Usa IA\n(gratuita)', 'Usa IA\n(pago próprio)', 'Empresa\npaga IA']
ia_vals   = [743, 2270, 274, 232]
ia_total  = 3772
ia_pcts   = [v/ia_total*100 for v in ia_vals]
ia_cores  = [ROSA, VERDE, AZUL_CLARO, AMARELO]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.patch.set_facecolor(AZUL_ESCURO)

ax1 = axes[0]; ax1.set_facecolor(AZUL_ESCURO)
wedges, texts, autotexts = ax1.pie(
    ia_pcts, labels=ia_labels, colors=ia_cores, autopct='%1.1f%%', startangle=140,
    textprops={'color': BRANCO, 'fontsize': 9},
    wedgeprops={'linewidth': 2, 'edgecolor': AZUL_ESCURO}
)
for at in autotexts: at.set_fontweight('bold')
ax1.set_title('Uso de IA Generativa — 2023', color=BRANCO, fontsize=12, fontweight='bold')

ax2 = axes[1]; ax2.set_facecolor(AZUL_ESCURO)
usa_pct = (2270+274+232)/3772*100
nao_pct = 743/3772*100
bars = ax2.barh(['Usa IA\n(qualquer forma)','Não usa IA'], [usa_pct, nao_pct], color=[VERDE, ROSA], alpha=0.9)
for bar, val in zip(bars, [usa_pct, nao_pct]):
    ax2.text(val+0.5, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}%', va='center', color=BRANCO, fontsize=13, fontweight='bold')
estilo(ax2, '% de Profissionais que Adotam IA\n(2023)', xlabel='%')
ax2.set_xlim(0, 105)
for l in ax2.get_yticklabels(): l.set_color(BRANCO)

plt.tight_layout()
plt.savefig('/tmp/grafico_ia.png', dpi=150, bbox_inches='tight', facecolor=AZUL_ESCURO)
plt.show()
subprocess.run(['aws','s3','cp','/tmp/grafico_ia.png',f'{BUCKET}/graficos/'])
print('Gráfico IA salvo!')
print('\n=== Análise concluída! Todos os gráficos exportados para S3. ===')